In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

T = 20
SCALE = 10.0

print("Device:", DEVICE)

Device: cpu


In [2]:
def step(s, a):
    s2 = s + 0.2 * a
    return s2, -np.abs(s2)


N = 2000

s = np.random.uniform(-1, 1, N)
q = np.random.rand(N)

S = np.zeros((N, T), np.float32)
A = np.zeros_like(S)
R = np.zeros_like(S)

for t in range(T):
    expert = -np.clip(s / 0.2, -1, 1)

    a = (1 - q) * expert + q * np.random.uniform(-1, 1, N)

    S[:, t], A[:, t] = s, a

    s, R[:, t] = step(s, a)

# Calculate Return-to-Go
RTG = np.cumsum(R[:, ::-1], 1)[:, ::-1].copy()

S_t = torch.tensor(S, device=DEVICE)[..., None]
A_t = torch.tensor(A, device=DEVICE)[..., None]
G_t = torch.tensor(RTG / SCALE, device=DEVICE)[..., None]

timesteps = torch.arange(T, device=DEVICE)

print("Dataset generated successfully!")
print("States shape:", S_t.shape)
print("Actions shape:", A_t.shape)
print("Returns shape:", G_t.shape)

Dataset generated successfully!
States shape: torch.Size([2000, 20, 1])
Actions shape: torch.Size([2000, 20, 1])
Returns shape: torch.Size([2000, 20, 1])


In [3]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d, heads, max_len):
        super().__init__()

        self.h = heads

        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(max_len, max_len)).bool()
        )

    def forward(self, x):

        B, L, D = x.shape

        q, k, v = self.qkv(x).split(D, dim=-1)

        q, k, v = [
            t.view(B, L, self.h, D // self.h).transpose(1, 2)
            for t in (q, k, v)
        ]

        att = (q @ k.transpose(-2, -1)) / (D // self.h) ** 0.5

        att = att.masked_fill(
            ~self.mask[:L, :L],
            float("-inf")
        )

        att = F.softmax(att, dim=-1)

        return self.proj(
            (att @ v).transpose(1, 2).reshape(B, L, D)
        )


print("Causal Self-Attention defined successfully!")

Causal Self-Attention defined successfully!


In [4]:
class Block(nn.Module):

    def __init__(self, d, heads, max_len):
        super().__init__()

        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)

        self.attn = CausalSelfAttention(d, heads, max_len)

        self.mlp = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.GELU(),
            nn.Linear(4 * d, d)
        )

    def forward(self, x):

        x = x + self.attn(self.ln1(x))

        return x + self.mlp(self.ln2(x))


print("Transformer Block created successfully!")

Transformer Block created successfully!


In [5]:
class DecisionTransformer(nn.Module):

    def __init__(self, d=32, heads=2, layers=2):
        super().__init__()

        self.emb_rtg = nn.Linear(1, d)
        self.emb_state = nn.Linear(1, d)
        self.emb_act = nn.Linear(1, d)

        self.emb_time = nn.Embedding(T, d)

        self.blocks = nn.Sequential(
            *[Block(d, heads, 3 * T) for _ in range(layers)]
        )

        self.ln = nn.LayerNorm(d)

        self.head = nn.Sequential(
            nn.Linear(d, 1),
            nn.Tanh()
        )

    def forward(self, rtg, states, actions, t):

        te = self.emb_time(t)[None]

        r = self.emb_rtg(rtg) + te
        s = self.emb_state(states) + te
        a = self.emb_act(actions) + te

        B, L, D = r.shape

        x = torch.stack([r, s, a], dim=2).reshape(B, 3 * L, D)

        x = self.ln(self.blocks(x)).reshape(B, L, 3, D)

        return self.head(x[:, :, 1])


model = DecisionTransformer().to(DEVICE)

print(model)
print("Decision Transformer created successfully!")

DecisionTransformer(
  (emb_rtg): Linear(in_features=1, out_features=32, bias=True)
  (emb_state): Linear(in_features=1, out_features=32, bias=True)
  (emb_act): Linear(in_features=1, out_features=32, bias=True)
  (emb_time): Embedding(20, 32)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=32, out_features=96, bias=True)
        (proj): Linear(in_features=32, out_features=32, bias=True)
      )
      (mlp): Sequential(
        (0): Linear(in_features=32, out_features=128, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=128, out_features=32, bias=True)
      )
    )
    (1): Block(
      (ln1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (at

In [6]:
opt = torch.optim.AdamW(
    model.parameters(),
    lr=2e-3
)

EPOCHS = 20
BATCH_SIZE = 256

loss_history = []

for epoch in range(EPOCHS):

    model.train()

    perm = torch.randperm(N, device=DEVICE)

    total_loss = 0

    for i in range(0, N, BATCH_SIZE):

        b = perm[i:i + BATCH_SIZE]

        pred = model(
            G_t[b],
            S_t[b],
            A_t[b],
            timesteps
        )

        loss = F.mse_loss(pred, A_t[b])

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()

    avg_loss = total_loss / ((N + BATCH_SIZE - 1) // BATCH_SIZE)

    loss_history.append(avg_loss)

    if epoch % 5 == 0 or epoch == 19:

        print(
            f"Epoch {epoch:2d} | Loss: {avg_loss:.4f}"
        )

print("Training completed successfully!")

Epoch  0 | Loss: 0.2028
Epoch  5 | Loss: 0.1317
Epoch 10 | Loss: 0.1143
Epoch 15 | Loss: 0.1070
Epoch 19 | Loss: 0.1043
Training completed successfully!


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(loss_history, marker='o')

plt.title("Decision Transformer Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.grid(True)
plt.show()

In [ ]:
@torch.no_grad()
def rollout(target, E=200):

    model.eval()

    s = torch.empty(E, 1, device=DEVICE).uniform_(-1, 1)

    rtg = torch.full((E, 1), target, device=DEVICE)

    total = torch.zeros(E, device=DEVICE)

    Ss, As, Gs = [], [], []

    for t in range(T):

        Ss.append(s)
        Gs.append(rtg / SCALE)
        As.append(torch.zeros(E, 1, device=DEVICE))

        Sb, Ab, Gb = (
            torch.stack(x, 1)
            for x in (Ss, As, Gs)
        )

        a = model(
            Gb,
            Sb,
            Ab,
            timesteps[:t + 1]
        )[:, -1]

        As[-1] = a

        s = s + 0.2 * a

        r = -s.abs()

        total += r.squeeze(1)

        rtg = rtg - r

    return total.mean().item()


print("Rollout function created successfully!")

In [ ]:
for target in [-8.0, -3.0, -0.5]:

    achieved = rollout(target)

    print(
        f"Target: {target:5.1f} -> Achieved: {achieved:.2f}"
    )

print("Evaluation completed!")